# Source Profiling: NYC 311

### Connection to the five analytical questions

- **Demand concentration:** `Borough`, `Created Date`, complaint type, coordinates and community-board fields.
- **Resolution performance:** `Created Date`, `Closed Date`, `Status`, agency and complaint type.
- **Repeated problems by property:** `BBL`, complaint type and creation date.
- **Unusually high activity vs similar properties:** a trustworthy 311-to-PLUTO BBL link, later combined with PLUTO attributes.
- **Variety of service needs by property type:** distinct complaint categories linked to PLUTO.

**Interpretation guardrails:** a BBL identifies a tax lot, not always a single building; several 311 requests can legitimately share one BBL. A missing `Closed Date` is expected for some open requests and must not automatically be treated as an error.

## Libraries and Spark session

In [2]:
import os
from datetime import date

import findspark
findspark.init()

from pyspark.sql import SparkSession

from pyspark.sql.functions import (
    col, lit, trim, length, count, countDistinct, coalesce, desc, when,
    try_to_timestamp, year,
    sum as spark_sum,
    min as spark_min,
    max as spark_max,
    round as spark_round
)
from pyspark.sql.types import (
    StructType, StructField,
    StringType, DoubleType
)

In [6]:
spark = (
    SparkSession.builder \
    .appName("DAT535-Project-NYC311") \
    .config("spark.sql.adaptive.enabled", "true") \
    .config("spark.sql.shuffle.partitions", "8") \
    .config("spark.driver.memory", "5g") \
    .getOrCreate()
)

spark.sparkContext.setLogLevel("WARN")
print("Spark version:", spark.version)

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/10/23 13:44:56 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Spark version: 3.5.0


## Check the DataSet file

In [32]:
PROJECT_DIR = "/home/ubuntu/spark-project-data"
BRONZE_DIR = f"{PROJECT_DIR}/bronze"
NYC311_PATH = f"{BRONZE_DIR}/nyc311/nyc311_2020_present.csv"

print("File exists:", os.path.isfile(NYC311_PATH))

File exists: True


## Define an explicit schema.

We keep `Unique Key`, `Created/Closed/Due Date` and `BBL` as **strings** so that we can inspect their original representation before casting them.

In [9]:

nyc311_explicit_schema = StructType([
    StructField("Unique Key", StringType(), True),
    StructField("Created Date", StringType(), True),
    StructField("Closed Date", StringType(), True),
    StructField("Agency", StringType(), True),
    StructField("Agency Name", StringType(), True),
    StructField("Problem (formerly Complaint Type)", StringType(), True),
    StructField("Problem Detail (formerly Descriptor)", StringType(), True),
    StructField("Additional Details", StringType(), True),
    StructField("Location Type", StringType(), True),
    StructField("Incident Zip", StringType(), True),
    StructField("Incident Address", StringType(), True),
    StructField("Street Name", StringType(), True),
    StructField("Cross Street 1", StringType(), True),
    StructField("Cross Street 2", StringType(), True),
    StructField("Intersection Street 1", StringType(), True),
    StructField("Intersection Street 2", StringType(), True),
    StructField("Address Type", StringType(), True),
    StructField("City", StringType(), True),
    StructField("Landmark", StringType(), True),
    StructField("Facility Type", StringType(), True),
    StructField("Status", StringType(), True),
    StructField("Due Date", StringType(), True),
    StructField("Resolution Description", StringType(), True),
    StructField("Resolution Action Updated Date", StringType(), True),
    StructField("Community Board", StringType(), True),
    StructField("Council District", StringType(), True),
    StructField("Police Precinct", StringType(), True),
    StructField("BBL", StringType(), True),
    StructField("Borough", StringType(), True),
    StructField("X Coordinate (State Plane)", StringType(), True),
    StructField("Y Coordinate (State Plane)", StringType(), True),
    StructField("Open Data Channel Type", StringType(), True),
    StructField("Park Facility Name", StringType(), True),
    StructField("Park Borough", StringType(), True),
    StructField("Vehicle Type", StringType(), True),
    StructField("Taxi Company Borough", StringType(), True),
    StructField("Taxi Pick Up Location", StringType(), True),
    StructField("Bridge Highway Name", StringType(), True),
    StructField("Bridge Highway Direction", StringType(), True),
    StructField("Road Ramp", StringType(), True),
    StructField("Bridge Highway Segment", StringType(), True),
    StructField("Latitude", DoubleType(), True),
    StructField("Longitude", DoubleType(), True),
    StructField("Location", StringType(), True)
])

expected_columns = [field.name for field in nyc311_explicit_schema.fields]
print("Expected schema column count:", len(expected_columns))

Expected schema column count: 44


## Verify Header

In [10]:

source_header_df = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "false")
    .csv(NYC311_PATH)
)
actual_columns = source_header_df.columns

print("Actual source column count:", len(actual_columns))
print("Header matches schema names and order:", actual_columns == expected_columns)

if actual_columns != expected_columns:
    missing_from_source = [name for name in expected_columns if name not in actual_columns]
    unexpected_in_source = [name for name in actual_columns if name not in expected_columns]
    print("Expected but absent from source:", missing_from_source)
    print("Unexpected source columns:", unexpected_in_source)
    print("Same set, different order:", set(actual_columns) == set(expected_columns))
    raise ValueError(
        "The NYC 311 header differs from the profiling schema. Review the actual header "
        "before reading rows with an explicit schema."
    )

Actual source column count: 44
Header matches schema names and order: True


## Create the DataFrame

In [12]:
NYC311_df = (
    spark.read
    .option("header", "true")
    .schema(nyc311_explicit_schema)
    .csv(NYC311_PATH)
)

## Inspect columns, types

In [13]:
print("Columns:")
print(NYC311_df.columns)
print("\nSchema:")
NYC311_df.printSchema()

Columns:
['Unique Key', 'Created Date', 'Closed Date', 'Agency', 'Agency Name', 'Problem (formerly Complaint Type)', 'Problem Detail (formerly Descriptor)', 'Additional Details', 'Location Type', 'Incident Zip', 'Incident Address', 'Street Name', 'Cross Street 1', 'Cross Street 2', 'Intersection Street 1', 'Intersection Street 2', 'Address Type', 'City', 'Landmark', 'Facility Type', 'Status', 'Due Date', 'Resolution Description', 'Resolution Action Updated Date', 'Community Board', 'Council District', 'Police Precinct', 'BBL', 'Borough', 'X Coordinate (State Plane)', 'Y Coordinate (State Plane)', 'Open Data Channel Type', 'Park Facility Name', 'Park Borough', 'Vehicle Type', 'Taxi Company Borough', 'Taxi Pick Up Location', 'Bridge Highway Name', 'Bridge Highway Direction', 'Road Ramp', 'Bridge Highway Segment', 'Latitude', 'Longitude', 'Location']

Schema:
root
 |-- Unique Key: string (nullable = true)
 |-- Created Date: string (nullable = true)
 |-- Closed Date: string (nullable = tru

## Example of records

In [15]:

NYC311_df.select(
    "Unique Key",
    "Created Date",
    "Closed Date",
    "Status",
    "Problem (formerly Complaint Type)",
    "Agency",
    "Borough",
    "BBL",
    "Community Board"
).show(15, truncate=False)

+----------+----------------------+-----------+-----------+---------------------------------+------+-------------+----------+----------------+
|Unique Key|Created Date          |Closed Date|Status     |Problem (formerly Complaint Type)|Agency|Borough      |BBL       |Community Board |
+----------+----------------------+-----------+-----------+---------------------------------+------+-------------+----------+----------------+
|70702319  |10/10/2026 03:03:53 AM|NULL       |Open       |Street Condition                 |DOT   |QUEENS       |NULL      |02 QUEENS       |
|70709493  |10/10/2026 02:06:05 AM|NULL       |In Progress|Noise - Residential              |NYPD  |QUEENS       |4037310065|05 QUEENS       |
|70703577  |10/10/2026 02:05:55 AM|NULL       |In Progress|Noise - Residential              |NYPD  |STATEN ISLAND|5070830014|03 STATEN ISLAND|
|70698764  |10/10/2026 02:05:53 AM|NULL       |In Progress|Blocked Driveway                 |NYPD  |QUEENS       |4092350019|09 QUEENS       |

## Select a reproducible sample for exploration

The following sample uses a fixed seed.

In [79]:
PROFILE_FRACTION = 0.01  # we keep 1% sample for exploration
PROFILE_FRACTION = 1.0  # test with full dataset
PROFILE_SEED = 535

NYC311_profile_df = NYC311_df.sample(
    withReplacement=False,
    fraction=PROFILE_FRACTION,
    seed=PROFILE_SEED
)

print(f"Profiling sample fraction: {PROFILE_FRACTION:.1%}")
print(f"Fixed sample seed: {PROFILE_SEED}")

Profiling sample fraction: 100.0%
Fixed sample seed: 535


## Record count and combined quality overview

The sample record count is calculated **once** in this combined aggregation and reused as the denominator for percentages. The date columns are parsed temporarily for profiling; the original DataFrame remains unchanged.

In [80]:
DATE_FORMAT = "MM/dd/yyyy hh:mm:ss a"
BBL_PATTERN = r"^[1-5][0-9]{9}$"  # 10 digits; borough code 1–5


def is_missing(column_name):
    """True for NULL or blank/whitespace-only values."""
    return col(column_name).isNull() | (trim(col(column_name).cast("string")) == "")


NYC311_dates_profile_df = (
    NYC311_profile_df
    .withColumn(
        "_created_ts",
        try_to_timestamp(col("Created Date"), lit(DATE_FORMAT))
    )
    .withColumn(
        "_closed_ts",
        try_to_timestamp(col("Closed Date"), lit(DATE_FORMAT))
    )
)

unique_key_missing = is_missing("Unique Key")
unique_key_invalid = (~unique_key_missing) & (~trim(col("Unique Key")).rlike(r"^[0-9]+$"))
created_missing = is_missing("Created Date")
closed_missing = is_missing("Closed Date")
created_invalid = (~created_missing) & col("_created_ts").isNull()
closed_invalid = (~closed_missing) & col("_closed_ts").isNull()
bbl_missing = is_missing("BBL")
bbl_valid = (~bbl_missing) & trim(col("BBL")).rlike(BBL_PATTERN)
bbl_invalid = (~bbl_missing) & (~trim(col("BBL")).rlike(BBL_PATTERN))
problem_missing = is_missing("Problem (formerly Complaint Type)")
status_missing = is_missing("Status")

profile_summary_row = NYC311_dates_profile_df.agg(
    count("*").alias("sample_record_count"),
    countDistinct(
        when(~problem_missing, trim(col("Problem (formerly Complaint Type)")))
    ).alias("distinct_nonblank_complaint_types"),
    countDistinct(when(~bbl_missing, trim(col("BBL")))).alias("distinct_nonmissing_bbls"),
    spark_sum(when(unique_key_missing, 1).otherwise(0)).alias("missing_unique_keys"),
    spark_sum(when(unique_key_invalid, 1).otherwise(0)).alias("invalid_unique_key_formats"),
    spark_sum(when(created_missing, 1).otherwise(0)).alias("missing_created_dates"),
    spark_sum(when(created_invalid, 1).otherwise(0)).alias("invalid_created_dates"),
    spark_sum(when(closed_missing, 1).otherwise(0)).alias("missing_closed_dates"),
    spark_sum(when(closed_invalid, 1).otherwise(0)).alias("invalid_closed_dates"),
    spark_sum(when(bbl_missing, 1).otherwise(0)).alias("missing_bbls"),
    spark_sum(when(bbl_invalid, 1).otherwise(0)).alias("invalid_nonmissing_bbls"),
    spark_sum(when(bbl_valid, 1).otherwise(0)).alias("valid_format_bbls"),
    spark_sum(when(status_missing, 1).otherwise(0)).alias("missing_status"),
    spark_sum(when(problem_missing, 1).otherwise(0)).alias("missing_complaint_type"),
    spark_sum(when(is_missing("Agency"), 1).otherwise(0)).alias("missing_agency"),
    spark_sum(when(is_missing("Borough"), 1).otherwise(0)).alias("missing_borough"),
    spark_sum(when(col("Latitude").isNull(), 1).otherwise(0)).alias("missing_latitude"),
    spark_sum(when(col("Longitude").isNull(), 1).otherwise(0)).alias("missing_longitude"),
    spark_sum(
        when(col("Latitude").isNull() | col("Longitude").isNull(), 1).otherwise(0)
    ).alias("records_missing_either_coordinate"),
    spark_min("_created_ts").alias("earliest_valid_created_at"),
    spark_max("_created_ts").alias("latest_valid_created_at"),
    spark_min("_closed_ts").alias("earliest_valid_closed_at"),
    spark_max("_closed_ts").alias("latest_valid_closed_at"),
    spark_sum(
        when(
            col("_created_ts").isNotNull()
            & col("_closed_ts").isNotNull()
            & (col("_closed_ts") < col("_created_ts")),
            1
        ).otherwise(0)
    ).alias("closed_before_created")
).first().asDict()

profile_record_count = int(profile_summary_row["sample_record_count"] or 0)



[Stage 211:===============================================>         (5 + 1) / 6]

sample_record_count: 22761196
distinct_nonblank_complaint_types: 276
distinct_nonmissing_bbls: 770925
missing_unique_keys: 0
invalid_unique_key_formats: 0
missing_created_dates: 0
invalid_created_dates: 0
missing_closed_dates: 439146
invalid_closed_dates: 0
missing_bbls: 2668612
invalid_nonmissing_bbls: 2440
valid_format_bbls: 20090144
missing_status: 0
missing_complaint_type: 0
missing_agency: 0
missing_borough: 38433
missing_latitude: 425647
missing_longitude: 425647
records_missing_either_coordinate: 425647
earliest_valid_created_at: 2020-01-01 00:00:00
latest_valid_created_at: 2026-10-10 03:03:53
earliest_valid_closed_at: 1899-12-31 19:00:00
latest_valid_closed_at: 2033-03-01 00:00:00
closed_before_created: 47207
valid-format BBL coverage (% of sample): 88.26
records with both coordinates (% of sample): 98.13


## Explore `Created Date` and `Closed Date`

In [81]:
DATE_FORMAT = "MM/dd/yyyy hh:mm:ss a"


def is_missing(column_name):
    """True for NULL or blank/whitespace-only values."""
    return (
        col(column_name).isNull()
        | (trim(col(column_name).cast("string")) == "")
    )


def count_if(condition):
    """Count records satisfying a condition."""
    return coalesce(
        spark_sum(when(condition, 1).otherwise(0)),
        lit(0)
    ).cast("long")


NYC311_dates_profile_df = (
    NYC311_profile_df
    .withColumn(
        "_created_ts",
        try_to_timestamp(col("Created Date"), lit(DATE_FORMAT))
    )
    .withColumn(
        "_closed_ts",
        try_to_timestamp(col("Closed Date"), lit(DATE_FORMAT))
    )
)

created_missing = is_missing("Created Date")
closed_missing = is_missing("Closed Date")

created_invalid = (
    (~created_missing) & col("_created_ts").isNull()
)

closed_invalid = (
    (~closed_missing) & col("_closed_ts").isNull()
)

### Check for `Created Dates`

In [82]:
created_date_profile = NYC311_dates_profile_df.agg(
    count_if(created_missing).alias("missing_created_dates"),
    count_if(created_invalid).alias("invalid_created_dates"),
    spark_min("_created_ts").alias("earliest_valid_created_at"),
    spark_max("_created_ts").alias("latest_valid_created_at")
)

created_date_profile.show(truncate=False)

[Stage 215:====================================================>(118 + 2) / 120]

+---------------------+---------------------+-------------------------+-----------------------+
|missing_created_dates|invalid_created_dates|earliest_valid_created_at|latest_valid_created_at|
+---------------------+---------------------+-------------------------+-----------------------+
|0                    |0                    |2020-01-01 00:00:00      |2026-10-10 03:03:53    |
+---------------------+---------------------+-------------------------+-----------------------+



### Check for `Closed Dates`

In [83]:
closed_date_profile = NYC311_dates_profile_df.agg(
    count_if(closed_missing).alias("missing_closed_dates"),
    count_if(closed_invalid).alias("invalid_closed_dates"),
    spark_min("_closed_ts").alias("earliest_valid_closed_at"),
    spark_max("_closed_ts").alias("latest_valid_closed_at")
)

closed_date_profile.show(truncate=False)

[Stage 218:====================================================>(119 + 1) / 120]

+--------------------+--------------------+------------------------+----------------------+
|missing_closed_dates|invalid_closed_dates|earliest_valid_closed_at|latest_valid_closed_at|
+--------------------+--------------------+------------------------+----------------------+
|439146              |0                   |1899-12-31 19:00:00     |2033-03-01 00:00:00   |
+--------------------+--------------------+------------------------+----------------------+



Some `Closed Date` values could be erroneous, with dates that are far too old (1899) or in the future (2033).

### Investigating erroneous `Closed Dates`

In [101]:
N = 10
DATE_FORMAT = "MM/dd/yyyy hh:mm:ss a"

closed_dates_df = (
    NYC311_profile_df
    .withColumn(
        "_closed_ts",
        try_to_timestamp(col("Closed Date"), lit(DATE_FORMAT))
    )
    .filter(col("_closed_ts").isNotNull())
)

columns_to_display = [
    "Unique Key",
    "Problem (formerly Complaint Type)",
    "Status",
    "Created Date",
    "Closed Date",
    "_closed_ts"
]

# N oldest closing dates
print(f"{N} requests with the oldest closing dates")

closed_dates_df.orderBy(
    col("_closed_ts").asc()
).select(
    *columns_to_display
).show(N, truncate=False)


# N most recent closing dates
print(f"{N} requests with the most recent closing dates")

closed_dates_df.orderBy(
    col("_closed_ts").desc()
).select(
    *columns_to_display
).show(N, truncate=False)

10 requests with the oldest closing dates


+----------+-----------------------------------+-----------+----------------------+----------------------+-------------------+
|Unique Key|Problem (formerly Complaint Type)  |Status     |Created Date          |Closed Date           |_closed_ts         |
+----------+-----------------------------------+-----------+----------------------+----------------------+-------------------+
|52552664  |Private or Charter School Reopening|In Progress|11/18/2021 03:33:44 AM|12/31/1899 07:00:00 PM|1899-12-31 19:00:00|
|46650421  |New Tree Request                   |In Progress|06/27/2020 04:21:15 PM|12/31/1899 07:00:00 PM|1899-12-31 19:00:00|
|47050142  |New Tree Request                   |In Progress|07/31/2020 01:47:14 PM|12/31/1899 07:00:00 PM|1899-12-31 19:00:00|
|46640681  |New Tree Request                   |In Progress|06/26/2020 08:04:02 PM|12/31/1899 07:00:00 PM|1899-12-31 19:00:00|
|49345918  |For Hire Vehicle Complaint         |In Progress|01/09/2021 07:12:51 AM|12/31/1899 07:00:00 PM|1899-

[Stage 314:====================================================>(118 + 2) / 120]

+----------+---------------------------------+------+----------------------+----------------------+-------------------+
|Unique Key|Problem (formerly Complaint Type)|Status|Created Date          |Closed Date           |_closed_ts         |
+----------+---------------------------------+------+----------------------+----------------------+-------------------+
|54304503  |Graffiti                         |Closed|05/28/2022 02:30:51 PM|03/01/2033 12:00:00 AM|2033-03-01 00:00:00|
|67996481  |Traffic Signal Condition         |Closed|02/14/2026 09:42:00 AM|12/14/2026 11:50:00 AM|2026-12-14 11:50:00|
|70703884  |Graffiti                         |Closed|10/09/2026 06:10:44 PM|10/10/2026 02:22:25 AM|2026-10-10 02:22:25|
|70703883  |Graffiti                         |Closed|10/09/2026 03:00:55 PM|10/10/2026 02:22:25 AM|2026-10-10 02:22:25|
|70708460  |Graffiti                         |Closed|10/09/2026 07:20:51 AM|10/10/2026 02:22:25 AM|2026-10-10 02:22:25|
|70700763  |Graffiti                    

The `Closing Dates` contain clear errors: several requests marked `In Progress` have a **closing date of 1899**, before their creation dates. This value of `12/31/1899 07:00:00 PM` is likely to be the default value of the software used to register NYC311 complains.
Two other records have future `Closing Dates` in December 2026 and March 2033. These values should be flagged for validation.

### Check for `Created Dates` anterior to `Closed Dates`

In [84]:
closed_before_created_profile = NYC311_dates_profile_df.agg(
    count_if(
        col("_created_ts").isNotNull()
        & col("_closed_ts").isNotNull()
        & (col("_closed_ts") < col("_created_ts"))
    ).alias("closed_before_created")
)

closed_before_created_profile.show(truncate=False)

[Stage 221:====================================================>(119 + 1) / 120]

+---------------------+
|closed_before_created|
+---------------------+
|47207                |
+---------------------+



A total of 47,207 records have a `Closed Date` earlier than their `Created Date`, indicating chronologically inconsistent dates that should be flagged for validation.

## Explore `Unique Key`

### Check for duplicate `Unique Key`

In [85]:
unique_keys_occurrences = (
    NYC311_profile_df
    .filter(~is_missing("Unique Key"))
    .withColumn("_unique_key_trim", trim(col("Unique Key")))
    .groupBy("_unique_key_trim")
    .agg(count("*").alias("occurrences"))
)

repeated_unique_keys = unique_keys_occurrences.filter(col("occurrences") > 1)
repeated_unique_keys_count = repeated_unique_keys.count()

if repeated_unique_keys_count == 0:
    print("No repeated Unique Keys were found.")
else:
    print(f"A total of {repeated_unique_keys_count} repeated Unique Keys were found.")

[Stage 226:======================================>                  (4 + 2) / 6]

No repeated Unique Keys were found.


### Check for missing `Unique Key` or wrong format

In [86]:
unique_key_missing = is_missing("Unique Key")

unique_key_invalid = (
    (~unique_key_missing)
    & (~trim(col("Unique Key")).rlike(r"^[0-9]+$"))
)

unique_key_profile = NYC311_profile_df.agg(
    count_if(unique_key_missing).alias("missing_unique_keys"),
    count_if(unique_key_invalid).alias("invalid_unique_key_formats")
)

unique_key_profile.show(truncate=False)

[Stage 230:====================================================>(118 + 2) / 120]

+-------------------+--------------------------+
|missing_unique_keys|invalid_unique_key_formats|
+-------------------+--------------------------+
|0                  |0                         |
+-------------------+--------------------------+



## Explore `BBL`

### Inspect `BBL` values

In [87]:
NYC311_profile_df.select("BBL").filter(~is_missing("BBL")).show(10, truncate=False)

+----------+
|BBL       |
+----------+
|4037310065|
|5070830014|
|4092350019|
|4092840019|
|3083000041|
|3066740023|
|3012820063|
|4054390060|
|3001460014|
|5033380056|
+----------+
only showing top 10 rows



### Check `BBL` Length and Format

BBL should be a 10-digit value beginning with borough code `1`–`5`.

In [88]:
BBL_PATTERN = r"^[1-5][0-9]{9}$"

bbl_format_profile = (
    NYC311_profile_df
    .withColumn("bbl_trim", trim(col("BBL")))
    .withColumn(
        "bbl_status",
        when(is_missing("BBL"), lit("missing"))
        .when(col("bbl_trim").rlike(BBL_PATTERN), lit("valid_format"))
        .otherwise(lit("invalid_format"))
    )
    .withColumn(
        "bbl_length",
        when(is_missing("BBL"), lit("missing"))
        .otherwise(length(col("bbl_trim")).cast("string"))
    )
)

bbl_format_profile.groupBy("bbl_status", "bbl_length").agg(
    count("*").alias("records")
).orderBy(desc("records")).show(30, truncate=False)

[Stage 234:====================================================>(118 + 2) / 120]

+--------------+----------+--------+
|bbl_status    |bbl_length|records |
+--------------+----------+--------+
|valid_format  |10        |20090144|
|missing       |missing   |2668612 |
|invalid_format|10        |2438    |
|invalid_format|2         |2       |
+--------------+----------+--------+



### Inspect `BBLs` with invalid Format

In [105]:
N = 10

invalid_bbl_df = bbl_format_profile.filter(
    col("bbl_status") == "invalid_format"
)

# Find all invalid BBL lengths
invalid_lengths = [
    row["bbl_length"]
    for row in (
        invalid_bbl_df
        .select("bbl_length")
        .distinct()
        .orderBy("bbl_length")
        .collect()
    )
]

# Display the N most frequent invalid BBL values for each length
for bbl_length in invalid_lengths:
    print(f"\nTop {N} invalid BBL values — length: {bbl_length}")

    (
        invalid_bbl_df
        .filter(col("bbl_length") == bbl_length)
        .groupBy("BBL", "bbl_length")
        .agg(count("*").alias("occurrences"))
        .orderBy(desc("occurrences"))
        .show(N, truncate=False)
    )


Top 10 invalid BBL values — length: 10


+----------+----------+-----------+
|BBL       |bbl_length|occurrences|
+----------+----------+-----------+
|0000000000|10        |2438       |
+----------+----------+-----------+


Top 10 invalid BBL values — length: 2


[Stage 332:====================================================>(118 + 2) / 120]

+---+----------+-----------+
|BBL|bbl_length|occurrences|
+---+----------+-----------+
|24 |2         |1          |
|14 |2         |1          |
+---+----------+-----------+



A total of 2,440 records have invalid `BBL` values: 2,438 contain 0000000000, which is not a valid `BBL`, and 2 contain only two digits (24 and 14).

### Total uniques `BBLs`

In [90]:
distinct_bbl_profile = NYC311_profile_df.agg(
    countDistinct(
        when(
            ~bbl_missing,
            trim(col("BBL"))
        )
    ).alias("distinct_nonmissing_bbls")
)

distinct_bbl_profile.show(truncate=False)

[Stage 242:=============================================>           (4 + 1) / 5]

+------------------------+
|distinct_nonmissing_bbls|
+------------------------+
|770925                  |
+------------------------+



## Explore `Borought`

In [103]:
def show_category_distribution(df, source_column, display_name, limit=15):
    category_df = (
        df.withColumn(
            "category",
            when(is_missing(source_column), lit("(missing)"))
            .otherwise(trim(col(source_column).cast("string")))
        )
        .groupBy("category")
        .agg(count("*").alias("records"))
        .orderBy(desc("records"))
    )
    print(f"\n{display_name} — top {limit}")
    category_df.show(limit, truncate=False)
    return category_df

borough_distribution = show_category_distribution(
    NYC311_profile_df, "Borough", "Borough", limit=15
)



Borough — top 15


[Stage 318:====================================================>(119 + 1) / 120]

+-------------+-------+
|category     |records|
+-------------+-------+
|BROOKLYN     |6833840|
|QUEENS       |5476485|
|BRONX        |4834551|
|MANHATTAN    |4580930|
|STATEN ISLAND|955909 |
|Unspecified  |41046  |
|(missing)    |38433  |
|Precinct 112 |1      |
|Precinct 52  |1      |
+-------------+-------+



`Precinct 112` and `Precinct 52` are **NYPD police precincts, not boroughs**. Their presence in the Borough column indicates a data quality issue. In Silver, these records should first be flagged as invalid borough values, then other fields should be checked to determine whether the borough can be corrected reliably.

Two records contain unexpected values (`Precinct 112` and `Precinct 52`) in the `Borough` column, which should be investigated.

## Explore `Address Type`

In [92]:
address_type_distribution = show_category_distribution(
    NYC311_profile_df, "Address Type", "Address type", limit=15
)


Address type — top 15


[Stage 249:====================================================>(118 + 2) / 120]

+------------+--------+
|category    |records |
+------------+--------+
|ADDRESS     |18101481|
|(missing)   |2801263 |
|INTERSECTION|1480500 |
|BLOCKFACE   |255647  |
|UNRECOGNIZED|72989   |
|PLACE       |36274   |
|PLACENAME   |13040   |
|BBL         |2       |
+------------+--------+



Two records have `Address Type` = BBL despite having no address information (see below), so these values should likely be treated as missing rather than valid address types.

In [93]:
NYC311_profile_df.filter(
    trim(col("Address Type")) == "BBL"
).select(
    "Unique Key",
    "Address Type",
    "Incident Address",
    "Street Name",
    "Incident Zip",
    "Borough",
    "BBL"
).show(truncate=False)

[Stage 255:=====================================================> (93 + 2) / 95]

+----------+------------+----------------+-----------+------------+--------+----------+
|Unique Key|Address Type|Incident Address|Street Name|Incident Zip|Borough |BBL       |
+----------+------------+----------------+-----------+------------+--------+----------+
|59835794  |BBL         |NULL            |NULL       |NULL        |QUEENS  |4118660010|
|59837458  |BBL         |NULL            |NULL       |NULL        |BROOKLYN|3072140014|
+----------+------------+----------------+-----------+------------+--------+----------+



## Explore `Location Type`

The `Location Type` column presents a lot of data quality issues:

- **Missing values:** 29,650 records have no value, limiting location-based analysis.
- **Missing values versus placeholders:** (missing) and N/A are separate categories, although both may indicate unavailable information.
- **Ambiguous or generic labels:** Above Address, Other (Explain Below), and N/A provide unclear information.
- **Inconsistent naming conventions:** Values such as RESIDENTIAL BUILDING and Residential Building/House use different capitalization.
- **Potentially overlapping categories:** Street/Sidewalk, Street, and Sidewalk, or Park and Park/Playground, or Restaurant/Bar/Deli/Bakery
and Club/Bar/Restaurant may overlap but are not equivalent.
- **Different levels of granularity:** Broad categories (Business, Store) coexist with more specific ones (Restaurant/Bar/Deli/Bakery).
- **Compound categories:** Labels such as Store/Commercial and Club/Bar/Restaurant combine multiple location types.

In [94]:
location_type_distribution = show_category_distribution(
    NYC311_profile_df, "Location Type", "Location type", limit=20
)


Location type — top 20


[Stage 256:===================================================> (117 + 3) / 120]

+----------------------------+-------+
|category                    |records|
+----------------------------+-------+
|Street/Sidewalk             |6521378|
|RESIDENTIAL BUILDING        |4486297|
|(missing)                   |2974990|
|Residential Building/House  |2771441|
|Street                      |2082960|
|Sidewalk                    |1654407|
|Store/Commercial            |422959 |
|Park                        |199434 |
|Club/Bar/Restaurant         |179318 |
|Above Address               |177534 |
|3+ Family Apt. Building     |121664 |
|Park/Playground             |113363 |
|Business                    |110170 |
|Subway                      |99084  |
|Restaurant/Bar/Deli/Bakery  |84862  |
|1-2 Family Dwelling         |64902  |
|3+ Family Apartment Building|51170  |
|Other (Explain Below)       |44744  |
|N/A                         |41701  |
|Commercial Building         |41199  |
+----------------------------+-------+
only showing top 20 rows



## Explore `Problem (formerly Complaint Type)`

sames issues as `Location Type` : inconsistent naming convention, duplicate categories (`Abandoned Vehicle` and `Derelict Vehicles`), overlaping categories (`General Construction/Plumbing` and `PLUMBING`)

### Explore the main values

In [95]:
complaint_distribution = show_category_distribution(
    NYC311_profile_df,
    "Problem (formerly Complaint Type)",
    "Complaint type",
    limit=25
)


Complaint type — top 25


[Stage 259:====================================================>(119 + 1) / 120]

+-----------------------------------+-------+
|category                           |records|
+-----------------------------------+-------+
|Illegal Parking                    |2964367|
|Noise - Residential                |2579719|
|HEAT/HOT WATER                     |1658600|
|Noise - Street/Sidewalk            |1177444|
|Blocked Driveway                   |1086462|
|UNSANITARY CONDITION               |716077 |
|Request Large Bulky Item Collection|632148 |
|Street Condition                   |519781 |
|Water System                       |422631 |
|Noise - Vehicle                    |417036 |
|PLUMBING                           |411056 |
|Abandoned Vehicle                  |406701 |
|Noise - Commercial                 |406432 |
|PAINT/PLASTER                      |375006 |
|Noise                              |348983 |
|Street Light Condition             |307647 |
|Derelict Vehicles                  |295500 |
|Traffic Signal Condition           |286338 |
|Dirty Condition                  

### Check the NULL and empty values

In [96]:
NYC311_profile_df.filter(
    col("Problem (formerly Complaint Type)").isNull()
    | (trim(col("Problem (formerly Complaint Type)")) == "")
).select(
    "Problem Detail (formerly Descriptor)",
    "Additional Details"
).show(truncate=False)

[Stage 265:=====================================================> (92 + 3) / 95]

+------------------------------------+------------------+
|Problem Detail (formerly Descriptor)|Additional Details|
+------------------------------------+------------------+
+------------------------------------+------------------+



## Explore `Agency`

In [97]:
agency_distribution = show_category_distribution(
    NYC311_profile_df, "Agency", "Agency", limit=150
)


Agency — top 150


[Stage 266:===================================================> (117 + 3) / 120]

+----------+-------+
|category  |records|
+----------+-------+
|NYPD      |9958135|
|HPD       |4499910|
|DSNY      |2545306|
|DOT       |1528270|
|DEP       |1213263|
|DPR       |868505 |
|DOB       |668583 |
|DOHMH     |549764 |
|DHS       |301868 |
|TLC       |205106 |
|EDC       |178490 |
|DCWP      |127055 |
|OSE       |70955  |
|DCA       |27905  |
|DOE       |8783   |
|OOS       |4705   |
|DFTA      |2605   |
|OTI       |947    |
|DOITT     |669    |
|NYC311-PRD|371    |
|3-1-1     |1      |
+----------+-------+



Agency codes should be **interpreted with caution** because some may reflect **historical naming changes** or reorganizations rather than distinct agencies. For example, `DCA` was renamed `DCWP` in 2020, and `DOITT` was consolidated into `OTI` in 2022.

The rare values `NYC311-PRD` (371 records) and `3-1-1` (1 record) should be investigated, because they are unusual values for an agency field.

## Explore `Status`

### Explore `Status` distribution

In [98]:
status_distribution = (
    NYC311_profile_df
    .withColumn(
        "_status",
        when(is_missing("Status"), lit("(missing)"))
        .otherwise(trim(col("Status")))
    )
    .groupBy("_status")
    .agg(count("*").alias("records"))
    .withColumn(
        "percentage_of_sample",
        spark_round(100 * col("records") / lit(profile_record_count), 2)
    )
    .orderBy(desc("records"))
)

status_distribution.show(truncate=False)

[Stage 269:====================================================>(119 + 1) / 120]

+-----------+--------+--------------------+
|_status    |records |percentage_of_sample|
+-----------+--------+--------------------+
|Closed     |22294838|97.95               |
|In Progress|286243  |1.26                |
|Open       |84638   |0.37                |
|Pending    |63306   |0.28                |
|Assigned   |24590   |0.11                |
|Started    |4739    |0.02                |
|Unspecified|2841    |0.01                |
|Cancel     |1       |0.0                 |
+-----------+--------+--------------------+



### Explore `Status` 

In [106]:
def show_status_profile(status):
    print(f"\nStatus: {status}")

    # Filters the rows
    status_df = NYC311_profile_df.filter(
        trim(col("Status")) == status
    )

    # Display 15 example records
    status_df.select(
        "Unique Key",
        "Problem (formerly Complaint Type)",
        "Created Date",
        "Due Date",
        "Closed Date"
    ).show(15, truncate=False)

    # Aggregate statistics for this status
    status_summary = status_df.agg(
        count("*").alias("total_records"),
        spark_sum(
            when(
                col("Created Date").isNotNull()
                & (trim(col("Created Date")) != ""),
                1
            ).otherwise(0)
        ).alias("records_with_created_date"),
        spark_sum(
            when(
                col("Due Date").isNotNull()
                & (trim(col("Due Date")) != ""),
                1
            ).otherwise(0)
        ).alias("records_with_due_date"),
        spark_sum(
            when(
                col("Closed Date").isNotNull()
                & (trim(col("Closed Date")) != ""),
                1
            ).otherwise(0)
        ).alias("records_with_closed_date")
    )

    status_summary.show(truncate=False)


# Run the profiling for each status
for status in [
    "Started",
    "In Progress",
    "Open",
    "Pending",
    "Assigned",
    "Unspecified",
    "Cancel"
]:
    show_status_profile(status)


Status: Started


+----------+---------------------------------+----------------------+--------+-----------+
|Unique Key|Problem (formerly Complaint Type)|Created Date          |Due Date|Closed Date|
+----------+---------------------------------+----------------------+--------+-----------+
|69869636  |Water System                     |07/28/2026 03:48:00 PM|NULL    |NULL       |
|69869638  |Noise                            |07/28/2026 04:49:00 AM|NULL    |NULL       |
|69840568  |Sewer                            |07/25/2026 02:35:00 PM|NULL    |NULL       |
|69841840  |Water System                     |07/25/2026 11:23:00 AM|NULL    |NULL       |
|69836560  |Sewer                            |07/25/2026 10:36:00 AM|NULL    |NULL       |
|69832152  |Water System                     |07/24/2026 05:20:00 PM|NULL    |NULL       |
|69832153  |Water System                     |07/24/2026 04:53:00 PM|NULL    |NULL       |
|69833605  |Water System                     |07/24/2026 02:31:00 PM|NULL    |NULL       |

+-------------+-------------------------+---------------------+------------------------+
|total_records|records_with_created_date|records_with_due_date|records_with_closed_date|
+-------------+-------------------------+---------------------+------------------------+
|4739         |4739                     |0                    |93                      |
+-------------+-------------------------+---------------------+------------------------+


Status: In Progress
+----------+---------------------------------+----------------------+--------+-----------+
|Unique Key|Problem (formerly Complaint Type)|Created Date          |Due Date|Closed Date|
+----------+---------------------------------+----------------------+--------+-----------+
|70709493  |Noise - Residential              |10/10/2026 02:06:05 AM|NULL    |NULL       |
|70703577  |Noise - Residential              |10/10/2026 02:05:55 AM|NULL    |NULL       |
|70698764  |Blocked Driveway                 |10/10/2026 02:05:53 AM|NULL    |

+-------------+-------------------------+---------------------+------------------------+
|total_records|records_with_created_date|records_with_due_date|records_with_closed_date|
+-------------+-------------------------+---------------------+------------------------+
|286243       |286243                   |0                    |2516                    |
+-------------+-------------------------+---------------------+------------------------+


Status: Open
+----------+---------------------------------+----------------------+--------+-----------+
|Unique Key|Problem (formerly Complaint Type)|Created Date          |Due Date|Closed Date|
+----------+---------------------------------+----------------------+--------+-----------+
|70702319  |Street Condition                 |10/10/2026 03:03:53 AM|NULL    |NULL       |
|70702953  |Street Light Condition           |10/10/2026 12:36:00 AM|NULL    |NULL       |
|70697673  |Street Condition                 |10/10/2026 12:20:32 AM|NULL    |NULL   

+-------------+-------------------------+---------------------+------------------------+
|total_records|records_with_created_date|records_with_due_date|records_with_closed_date|
+-------------+-------------------------+---------------------+------------------------+
|84638        |84638                    |1809                 |13387                   |
+-------------+-------------------------+---------------------+------------------------+


Status: Pending
+----------+---------------------------------+----------------------+--------+----------------------+
|Unique Key|Problem (formerly Complaint Type)|Created Date          |Due Date|Closed Date           |
+----------+---------------------------------+----------------------+--------+----------------------+
|70700739  |Street Condition                 |10/08/2026 10:40:19 PM|NULL    |NULL                  |
|70691411  |Street Condition                 |10/08/2026 01:25:48 PM|NULL    |NULL                  |
|70689901  |Street Conditio

...

+----------+---------------------------------+----------------------+--------+----------------------+
|Unique Key|Problem (formerly Complaint Type)|Created Date          |Due Date|Closed Date           |
+----------+---------------------------------+----------------------+--------+----------------------+
|70700892  |Noise                            |10/09/2026 11:30:26 PM|NULL    |NULL                  |
|70705503  |Water Maintenance                |10/09/2026 12:24:00 PM|NULL    |NULL                  |
|70694819  |Noise                            |10/08/2026 06:23:21 AM|NULL    |10/09/2026 12:00:00 PM|
|70690139  |Noise                            |10/08/2026 02:15:26 AM|NULL    |10/09/2026 01:10:00 PM|
|70676504  |Homeless Person Assistance       |10/07/2026 06:56:43 PM|NULL    |NULL                  |
|70653541  |Noise                            |10/05/2026 10:38:03 PM|NULL    |10/07/2026 04:43:00 PM|
|70656606  |Noise                            |10/05/2026 09:44:26 AM|NULL    |10/0

+-------------+-------------------------+---------------------+------------------------+
|total_records|records_with_created_date|records_with_due_date|records_with_closed_date|
+-------------+-------------------------+---------------------+------------------------+
|2841         |2841                     |0                    |2740                    |
+-------------+-------------------------+---------------------+------------------------+


Status: Cancel


+----------+---------------------------------+----------------------+--------+-----------+
|Unique Key|Problem (formerly Complaint Type)|Created Date          |Due Date|Closed Date|
+----------+---------------------------------+----------------------+--------+-----------+
|48260400  |Highway Condition                |01/21/2020 07:58:03 PM|NULL    |NULL       |
+----------+---------------------------------+----------------------+--------+-----------+



[Stage 366:====================================================>(119 + 1) / 120]

+-------------+-------------------------+---------------------+------------------------+
|total_records|records_with_created_date|records_with_due_date|records_with_closed_date|
+-------------+-------------------------+---------------------+------------------------+
|1            |1                        |0                    |0                       |
+-------------+-------------------------+---------------------+------------------------+



- Some of the records having the status **In Progress**, **Open**, **Pending**, **Assigned** have a `Closing Date` despite not having a Closed Status, indicating a mismatch between the Status and the Dates.

- **Open:** Some records have identical `Created Date` and `Closed Date` which should not be possible.

- **Pending:** Some records have a `Closed Date` earlier than their `Created Date`, which is impossible and should be flagged for validation.

- **Unspecified:** Some records lack a proper `Status` value, despite most of them having a `Closed Date`.

- **Cancel:** Only one record contains the `Status` **Cancel**.

## NYC 311 findings to carry forward

- **Source snapshot:** actual file path, file size, source URL, and real download/snapshot date.
- **Schema:** confirm 44 expected columns and note any header mismatch or schema change.
- **Profiling scope:** fixed-seed 1% sample (`seed=535`); record the sample row count and note that it is not the total source count.
- **Request ID (`Unique Key`):** missing/blank count, nonnumeric-format count, and repeated-ID metrics. Repeat ID checks exhaustively on the full data during Silver quality validation.
- **Creation/closure dates:** observed timestamp format and min/max parsed dates; distinguish missing dates from invalid nonempty strings; retain legitimate open requests.
- **BBL linkage:** valid 10-digit BBL coverage, missing/invalid BBL counts and formats. Compare those results with Member B's PLUTO profile before agreeing on normalization.
- **Categories and status:** leading complaint types, complaint types values issues, agencies and status categories. All distribution counts in this notebook are sample-based.
- **Geographic coverage:** borough distribution, address/location types.
- **Silver questions to agree together:** column mappings; status normalization; date parsing/reversal rules; treatment of unusable Unique Keys; BBL normalization; and how to account for duplicate/quarantined records.